In [ ]:
import numpy as np
from umap import UMAP
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from torchvision.datasets import MNIST, CIFAR10

mnist = MNIST(root='.', train=True, download=True)
cifar10 = CIFAR10(root='.', train=True, download=True)

In [ ]:
def convert_sklearn_dataset(pytorch_dataset):
  X, y = [], []
  for image, label in pytorch_dataset:
    x = np.array(image)
    x = x / 255
    X.append(x)
    y.append(label)
  X = np.array(X)
  X = X.reshape(len(X), -1) # 이미지를 1차원 벡터로 평탄화(MAP, t-SNE 같은 함수에 넣을 수 있도록)
  y = np.array(y)
  return X, y

In [ ]:
mnist_X, mnist_y = convert_sklearn_dataset(mnist)
cifar10_X, cifar10_y = convert_sklearn_dataset(cifar10)

In [ ]:
mnist_X
# array([[0., 0., 0., ..., 0., 0., 0.],
#        [0., 0., 0., ..., 0., 0., 0.],
#        [0., 0., 0., ..., 0., 0., 0.],
#        ...,
#        [0., 0., 0., ..., 0., 0., 0.],
#        [0., 0., 0., ..., 0., 0., 0.],
#        [0., 0., 0., ..., 0., 0., 0.]], shape=(60000, 784))

In [ ]:
tsne_random = TSNE(n_components=2, perplexity=30, init="random", random_state=2026)
tsne_pca = TSNE(n_components=2, perplexity=30, init="pca", random_state=2026)

np.random.seed(2026)
mnist_idx = np.random.choice(len(mnist_X), 1000, replace=False)
cifar10_idx = np.random.choice(len(cifar10_X), 1000, replace=False)

In [ ]:
def plot_embedding(model, X, y, idx):
    X_set = X[idx]
    y_set = y[idx]

    X_set = model.fit_transform(X_set)
    class_names = set(y_set)

    for i, class_name in enumerate(class_names):
        plt.scatter(
            X_set[y_set == class_name, 0],
            X_set[y_set == class_name, 1],
            color=plt.cm.tab10(i),
            label=class_name,
        )
    plt.xlabel('component 0')
    plt.ylabel('component 1')
    plt.legend()
    plt.show()

In [ ]:
plot_embedding(tsne_random, mnist_X, mnist_y, mnist_idx)

In [ ]:
plot_embedding(tsne_pca, mnist_X, mnist_y, mnist_idx)

In [ ]:
plot_embedding(tsne_random, cifar10_X, cifar10_y, cifar10_idx)

In [ ]:
plot_embedding(tsne_pca, cifar10_X, cifar10_y, cifar10_idx)

In [ ]:
umap = UMAP(n_components=2, min_dist=.05, n_neighbors=8, random_state=2025)

plot_embedding(umap, mnist_X, mnist_y, mnist_idx)

In [ ]:
plot_embedding(umap, cifar10_X, cifar10_y, cifar10_idx)

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

feature_extractor = resnet18(weights=ResNet18_Weights.DEFAULT)
feature_extractor.fc = nn.Identity()

In [ ]:
import torch
from torchvision.transforms import v2

transforms = v2.Compose([
    v2.ToImage(),
    v2.Resize(size=(96, 96), antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
cifar10_imagenet = CIFAR10(root='.', train=True, download=True, transform=transforms)

In [ ]:
cifar10_imagenet_X = torch.stack([cifar10_imagenet[idx][0] for idx in cifar10_idx])
cifar10_imagenet_y = np.array([cifar10_imagenet[idx][1] for idx in cifar10_idx])
feature_X = feature_extractor(cifar10_imagenet_X).detach().numpy()

In [ ]:
plot_embedding(tsne_pca, feature_X, cifar10_imagenet_y, list(range(len(cifar10_imagenet_y))))

In [ ]:
!pip install medmnist

In [ ]:
import medmnist
print(f"MedMNIST v{medmnist.__version__}")

In [ ]:
from medmnist import INFO

pathmnist_info = INFO['pathmnist']
DataClass = getattr(medmnist, pathmnist_info['python_class'])

In [ ]:
pathmnist = DataClass(split='train', download=True)
pathmnist.montage(length=20)

In [ ]:
pathmnist_X, pathmnist_y = convert_sklearn_dataset(pathmnist)
pathmnist_y = pathmnist_y[:, 0]

In [ ]:
pathmnist_idx = np.random.choice(len(pathmnist_X), 1000, replace=False)

plot_embedding(tsne_random, pathmnist_X, pathmnist_y, pathmnist_idx)

In [ ]:
plot_embedding(tsne_pca, pathmnist_X, pathmnist_y, pathmnist_idx)

In [ ]:
pathmnist_imagenet = DataClass(split='train', download=True, transform=transforms)
pathmnist_imagenet_X = torch.stack([pathmnist_imagenet[idx][0] for idx in pathmnist_idx])
pathmnist_imagenet_y = np.array([pathmnist_imagenet[idx][1] for idx in pathmnist_idx])[:, 0]
feature_X = feature_extractor(pathmnist_imagenet_X).detach().numpy()

In [ ]:
plot_embedding(tsne_pca, feature_X, pathmnist_imagenet_y, list(range(len(pathmnist_imagenet_y))))